<a href="https://colab.research.google.com/github/arulesharulesh/nextpredict/blob/main/nextword.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re, random
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

random.seed(42); np.random.seed(42); tf.random.set_seed(42)

# Read the book
text = open("/content/1661-0.txt", encoding="utf-8-sig").read()

# Keep only the story (remove Gutenberg header/footer)
start = text.index("*** START")
end   = text.index("*** END")
text  = text[start:end].split("***", 2)[2]

# Clean: fix quotes, remove line breaks and extra spaces
text = (text.replace("\r", " ").replace("’", "'")
            .replace("“", '"').replace("”", '"'))
text = re.sub(r"\s+", " ", text)

# Split into sentences and drop very short ones
sentences = re.split(r"(?<=[.!?])\s+", text)
sentences = [s for s in sentences if len(s.split()) >= 3]

print("Total sentences:", len(sentences))
print("Example:", sentences[100])

Total sentences: 4903
Example: There's money in this case, Watson, if there is nothing else." "I think that I had better go, Holmes." "Not a bit, Doctor.


In [2]:
random.shuffle(sentences)
split = int(0.9 * len(sentences))
train_sents, test_sents = sentences[:split], sentences[split:]

VOCAB_SIZE = 5000                      # keep the 5000 most frequent words

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<unk>")
tokenizer.fit_on_texts(train_sents)    # builds the vocabulary

print("Unique words found:", len(tokenizer.word_index))
print("First 10 vocab items:", list(tokenizer.word_index.items())[:10])
print("Sentence as numbers:", tokenizer.texts_to_sequences([train_sents[0]])[0])

Unique words found: 7907
First 10 vocab items: [('<unk>', 1), ('the', 2), ('and', 3), ('i', 4), ('to', 5), ('of', 6), ('a', 7), ('in', 8), ('that', 9), ('it', 10)]
Sentence as numbers: [293, 27, 9, 1015, 13, 4, 229, 18, 143, 70, 107, 932, 94, 13, 620, 241, 39, 690, 85, 9, 307, 44, 10, 12, 2943, 5, 2, 434]


In [3]:
def make_sequences(sents):
    X_raw, y = [], []
    for s in sents:
        tokens = tokenizer.texts_to_sequences([s])[0]
        for i in range(1, len(tokens)):
            target = tokens[i]
            if target == 1:            # skip "<unk>" targets (useless to predict)
                continue
            X_raw.append(tokens[:i])   # words typed so far
            y.append(target)           # the next word
    return X_raw, y

Xtr_raw, ytr = make_sequences(train_sents)
Xte_raw, yte = make_sequences(test_sents)

MAX_LEN = 20   # use at most the last 20 words as context
Xtr = pad_sequences(Xtr_raw, maxlen=MAX_LEN, padding="pre")
Xte = pad_sequences(Xte_raw, maxlen=MAX_LEN, padding="pre")
ytr, yte = np.array(ytr), np.array(yte)

print("Training examples:", Xtr.shape, " Test examples:", Xte.shape)
print("Example input :", Xtr[5])
print("Example target:", ytr[5], "=", tokenizer.index_word[ytr[5]])

Training examples: (87615, 20)  Test examples: (9382, 20)
Example input : [   0    0    0    0    0    0    0    0    0    0    0    0    0    0
  293   27    9 1015   13    4]
Example target: 229 = cannot


In [4]:
def build_model():
    model = Sequential([
        Embedding(VOCAB_SIZE, 100),            # word -> 100-number vector
        LSTM(150),                             # reads the word sequence
        Dropout(0.3),                          # reduces overfitting
        Dense(VOCAB_SIZE, activation="softmax")  # probability of each next word
    ])
    model.compile(
        loss="sparse_categorical_crossentropy",   # targets are word ids
        optimizer="adam",
        metrics=["accuracy",
                 tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5")]
    )
    return model

model = build_model()
model.build(input_shape=(None, MAX_LEN))
model.summary()

early = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)

history = model.fit(Xtr, ytr,
                    validation_split=0.1,
                    epochs=20, batch_size=128,
                    callbacks=[early], verbose=1)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 20, 100)        │       500,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 150)            │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 150)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5000)           │       755,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,405,600 (5.36 MB)

 Trainable params: 1,405,600 (5.36 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
617/617 ━━━━━━━━━━━━━━━━━━━━ 46s 72ms/step - accuracy: 0.0597 - loss: 6.3285 - top5: 0.1692 - val_accuracy: 0.0861 - val_loss: 6.0056 - val_top5: 0.2081
Epoch 2/20
617/617 ━━━━━━━━━━━━━━━━━━━━ 43s 69ms/step - accuracy: 0.0915 - loss: 5.8308 - top5: 0.2350 - val_accuracy: 0.1110 - val_loss: 5.6800 - val_top5: 0.2638
Epoch 3/20
617/617 ━━━━━━━━━━━━━━━━━━━━ 42s 68ms/step - accuracy: 0.1156 - loss: 5.5176 - top5: 0.2805 - val_accuracy: 0.1236 - val_loss: 5.5127 - val_top5: 0.2894
Epoch 4/20
617/617 ━━━━━━━━━━━━━━━━━━━━ 44s 71ms/step - accuracy: 0.1292 - loss: 5.3149 - top5: 0.3009 - val_accuracy: 0.1285 - val_loss: 5.4327 - val_top5: 0.2984
Epoch 5/20
617/617 ━━━━━━━━━━━━━━━━━━━━ 44s 71ms/step - accuracy: 0.1363 - loss: 5.1616 - top5: 0.3161 - val_accuracy: 0.1340 - val_loss: 5.3835 - val_top5: 0.3022
Epoch 6/20
617/617 ━━━━━━━━━━━━━━━━━━━━ 80s 68ms/step - accuracy: 0.1437 - loss: 5.0296 - top5: 0.3265 - val_accuracy: 0.1391 - val_loss: 5.3451 - val_top5: 0.3118
Epoch 7/20
617/6

In [5]:
loss, top1, top5 = model.evaluate(Xte, yte, verbose=0)
print(f"Test Top-1 accuracy: {top1*100:.2f}%")
print(f"Test Top-5 accuracy: {top5*100:.2f}%")

Test Top-1 accuracy: 14.88%
Test Top-5 accuracy: 32.99%


In [6]:
def predict_next(phrase, k=5):
    tokens = tokenizer.texts_to_sequences([phrase])[0]
    padded = pad_sequences([tokens], maxlen=MAX_LEN, padding="pre")
    probs  = model.predict(padded, verbose=0)[0]
    probs[[0, 1]] = 0                       # ignore padding and <unk>
    top_ids = probs.argsort()[-k:][::-1]    # k highest probabilities
    return [(tokenizer.index_word[i], round(float(probs[i]), 3)) for i in top_ids]

test_phrases = [
    "sherlock",
    "i have no doubt that",
    "it is a capital mistake to",
    "my dear",
    "he took out his",
    "there was a knock at the",
]

for p in test_phrases:
    print(f"\nInput : '{p}'")
    for word, prob in predict_next(p):
        print(f"   -> {word:12s} {prob}")


Input : 'sherlock'
   -> holmes       0.791
   -> he           0.046
   -> i            0.017
   -> the          0.015
   -> that         0.014

Input : 'i have no doubt that'
   -> i            0.221
   -> you          0.101
   -> it           0.079
   -> he           0.076
   -> the          0.061

Input : 'it is a capital mistake to'
   -> be           0.113
   -> the          0.054
   -> me           0.034
   -> have         0.031
   -> a            0.028

Input : 'my dear'
   -> is           0.035
   -> young        0.023
   -> and          0.016
   -> little       0.015
   -> very         0.014

Input : 'he took out his'
   -> head         0.044
   -> eyes         0.029
   -> hands        0.026
   -> face         0.025
   -> chair        0.023

Input : 'there was a knock at the'
   -> matter       0.02
   -> morning      0.015
   -> house        0.012
   -> time         0.01
   -> other        0.01


In [7]:
print(predict_next("what do you"))

[('know', 0.062), ('have', 0.053), ('see', 0.033), ('not', 0.03), ('think', 0.029)]


In [9]:
print(predict_next("hello how are "))

[('the', 0.065), ('a', 0.061), ('it', 0.04), ('as', 0.028), ('i', 0.025)]


In [11]:
print(predict_next("i love "))

[('that', 0.207), ('it', 0.109), ('the', 0.065), ('you', 0.058), ('to', 0.054)]


In [12]:
print(predict_next("i want to "))

[('be', 0.066), ('see', 0.045), ('do', 0.044), ('you', 0.029), ('have', 0.025)]
